In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

In [2]:
import os, cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
base_path = './DATASET'

lr_path = os.path.join(base_path,"train_Low_Resolution")
hr_path = os.path.join(base_path,"train_High_Resolution")
test_lr_path= os.path.join(base_path,"test_Low_Resolution")

In [5]:
#DATASET

class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, augment=False):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.files  = sorted(os.listdir(lr_dir))
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]

        lr = cv2.cvtColor(cv2.imread(os.path.join(self.lr_dir, name)), cv2.COLOR_BGR2RGB)
        hr = cv2.cvtColor(cv2.imread(os.path.join(self.hr_dir, name)), cv2.COLOR_BGR2RGB)

        if self.augment:
            if np.random.rand()>0.5:
                lr = cv2.flip(lr,1)
                hr = cv2.flip(hr,1)
            if np.random.rand() >0.5:
                lr = cv2.flip(lr,0)
                hr = cv2.flip(hr,0)
        lr = torch.from_numpy(lr.astype(np.float32)/255.0).permute(2,0,1)
        hr = torch.from_numpy(hr.astype(np.float32)/255.0).permute(2,0,1)
        return lr, hr

train_loader = DataLoader(SRDataset(lr_path, hr_path, True), batch_size=8, shuffle=True)
eval_loader  = DataLoader(SRDataset(lr_path, hr_path, False), batch_size=8, shuffle=False)


In [6]:
#LOSS FUNCTION 

class GradientLoss(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],dtype=torch.float32)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)
        self.kx = kx.view(1,1,3,3)
        self.ky = ky.view(1,1,3,3)

    def forward(self,pred,target):
        self.kx =self.kx.to(pred.device)
        self.ky = self.ky.to(pred.device)

        loss = 0
        for c in range(3):
            p = pred[:,c:c+1]
            t = target[:,c:c+1]

            loss += F.l1_loss(F.conv2d(p,self.kx,padding=1),
                              F.conv2d(t,self.kx,padding=1))
            loss += F.l1_loss(F.conv2d(p,self.ky,padding=1),
                              F.conv2d(t,self.ky,padding=1))

        return loss/3


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.grad = GradientLoss()

    def forward(self,pred,target):
        charb = torch.mean(torch.sqrt((pred -target)**2 +1e-6))   #Used Charbonnier loss 
        l1    = F.l1_loss(pred, target)
        grad  = self.grad(pred, target)
        return charb + 0.5*l1 + 0.1*grad


In [7]:
class ResidualBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch,3,padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.PReLU(),
            nn.Conv2d(ch, ch,3,padding=1, bias=False),
            nn.BatchNorm2d(ch)
        )

    def forward(self,x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.entry = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1),
            nn.PReLU()
        )

        self.res = nn.Sequential(*[ResidualBlock(64) for _ in range(16)])

        self.mid = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64)
        )

        self.up = nn.Sequential(
            nn.Conv2d(64,256,3,padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(64,256,3,padding=1),
            nn.PixelShuffle(2),
            nn.PReLU()
        )

        self.out = nn.Conv2d(64,3,3,padding=1)

    def forward(self, x):
        f= self.entry(x)
        r =self.res(f)
        r=self.mid(r) + f
        r = self.up(r)
        return self.out(r)


In [8]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6,64,3,1,1),nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,3,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(0.2),
            nn.Conv2d(128,256,3,2,1),nn.BatchNorm2d(256),nn.LeakyReLU(0.2),
            nn.Conv2d(256,512,3,2,1),nn.BatchNorm2d(512),nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512,1)
        )

    def forward(self, sr, lr):
        return self.net(torch.cat([sr, lr], dim=1))


In [9]:
def eval_mae(model, loader):
    model.eval()
    err, count = 0, 0

    with torch.no_grad():
        for lr, hr in loader:
            lr, hr= lr.to(device), hr.to(device)
            sr = model(lr)

            err += torch.sum(torch.abs(sr*255 - hr*255)).item()
            count += sr.numel()

    return err / count

In [10]:
G = Generator().to(device)
D = Discriminator().to(device)

optG = torch.optim.Adam(G.parameters(),lr=2e-4)
optD = torch.optim.Adam(D.parameters(),lr=1e-5)

loss_fn = CombinedLoss()
bce = nn.BCEWithLogitsLoss()

best_mae = float("inf")
best_weights = None


for ep in range(90):
    G.train()
    D.train()

    total_g_loss=0
    total_d_loss=0
    steps=0

    for lr, hr in train_loader:
        lr, hr = lr.to(device), hr.to(device)
        bs = lr.size(0)
        lr_up = F.interpolate(lr, size=(128,128), mode='bicubic')

        real = torch.ones(bs,1).to(device)
        fake = torch.zeros(bs,1).to(device)

        #Discriminator
        with torch.no_grad():
            fake_img = torch.clamp(G(lr), 0, 1)

        d_real = bce(D(hr, lr_up), real * 0.9)
        d_fake = bce(D(fake_img, lr_up), fake + 0.1)
        d_loss = (d_real+d_fake)/2

        optD.zero_grad()
        d_loss.backward()
        optD.step()

        #Generator 
        sr = torch.clamp(G(lr), 0, 1)

        adv = bce(D(sr, lr_up), real)
        pix = loss_fn(sr, hr)

        if ep < 60:
            g_loss = pix
            gan_flag = "gentraining"
        else:
            g_loss = pix + 0.001 * adv
            gan_flag = "gan"

        optG.zero_grad()
        g_loss.backward()
        optG.step()

        total_g_loss += g_loss.item()
        total_d_loss += d_loss.item()
        steps += 1

    #averages
    avg_g = total_g_loss/steps
    avg_d = total_d_loss/steps

    mae = eval_mae(G, eval_loader)

    print(f"Epoch {ep:02d} | G: {avg_g:.4f} | D: {avg_d:.4f} | MAE: {mae:.4f} | mode: {gan_flag}")

    
    if avg_d < 0.1:
        print("D very strong")
    elif avg_d > 1.0:
        print("G dominating or D weak")

    
 

Epoch 00 | G: 0.1942 | D: 0.6666 | MAE: 20.6401 | mode: gentraining
Epoch 01 | G: 0.1608 | D: 0.5736 | MAE: 18.6228 | mode: gentraining
Epoch 02 | G: 0.1577 | D: 0.5033 | MAE: 18.8666 | mode: gentraining
Epoch 03 | G: 0.1555 | D: 0.4575 | MAE: 18.1121 | mode: gentraining
Epoch 04 | G: 0.1542 | D: 0.4257 | MAE: 17.9015 | mode: gentraining
Epoch 05 | G: 0.1530 | D: 0.4000 | MAE: 17.8102 | mode: gentraining
Epoch 06 | G: 0.1530 | D: 0.3821 | MAE: 17.8269 | mode: gentraining
Epoch 07 | G: 0.1524 | D: 0.3693 | MAE: 17.9224 | mode: gentraining
Epoch 08 | G: 0.1519 | D: 0.3597 | MAE: 17.6538 | mode: gentraining
Epoch 09 | G: 0.1513 | D: 0.3503 | MAE: 17.6898 | mode: gentraining
Epoch 10 | G: 0.1511 | D: 0.3445 | MAE: 17.6711 | mode: gentraining
Epoch 11 | G: 0.1508 | D: 0.3397 | MAE: 17.5412 | mode: gentraining
Epoch 12 | G: 0.1505 | D: 0.3362 | MAE: 17.6187 | mode: gentraining
Epoch 13 | G: 0.1500 | D: 0.3342 | MAE: 17.4581 | mode: gentraining
Epoch 14 | G: 0.1499 | D: 0.3321 | MAE: 17.6047 

In [11]:

G.eval()

rows = []
files = sorted(os.listdir(test_lr_path))  #keeping order consistent

with torch.no_grad():
    for name in files:
        # read LR image
        lr = cv2.imread(os.path.join(test_lr_path, name))
        lr = cv2.cvtColor(lr, cv2.COLOR_BGR2RGB)

        # convert to tensor
        lr = torch.from_numpy(lr.astype(np.float32) / 255.0)
        lr = lr.permute(2, 0, 1).unsqueeze(0).to(device)

        # run through model
        sr = G(lr)
        sr = torch.clamp(sr, 0, 1).squeeze(0).cpu()

        # back to image format
        sr = sr.permute(1, 2, 0).numpy()
        sr = (sr * 255).clip(0, 255).astype(np.uint8)

        # flatten and store
        pixel_str = " ".join(sr.flatten().astype(str))

        rows.append({
            "Id": name,
            "Pixels": pixel_str
        })




In [12]:
submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

In [13]:
submission.head()
submission.shape

(495, 2)